# Audit Layer - Observability & Compliance

**Purpose**: Comprehensive audit logging for pipeline operations, data quality, access control, and compliance reporting.

**Target Audience**: DataOps engineers, compliance teams, security analysts

**Layer Position**: Cross-cutting observability layer monitoring all other layers (Bronze, Silver, Semantic, Warehouse, Gold)

---

## 🔍 Overview

The Audit layer provides **operational visibility and regulatory compliance** through:
* **Pipeline Run Tracking** - Complete execution history with success/failure details
* **Data Quality Monitoring** - DQ rule violations and trend analysis
* **Access Control Auditing** - Security events and denied access attempts
* **Operational Reporting** - Daily summaries and SLA compliance tracking
* **Alert Dispatching** - Real-time notifications for critical events

### Architecture Principles

* **Immutable Audit Log** - All events are append-only, never updated
* **Comprehensive Context** - Every log includes user, timestamp, environment, and correlation IDs
* **Retention Policies** - Automated cleanup based on compliance requirements
* **Multi-Channel Alerting** - Email, Slack, webhook integrations for notifications
* **Privacy First** - PII scrubbing in logs, secure credential management

---

## 📁 Audit Notebooks

### **Core Audit Logging**

#### 1. `audit_pipeline_runs`
**Purpose**: Log and track all pipeline execution runs  
**Target Table**: `workspace.audit.audit_pipeline_runs`  
**Captures**: Run ID, pipeline name, environment, start/end times, status, error details, records processed, user  
**Triggers**: Called by all pipeline notebooks via Pipeline_Metadata_Manager  
**Retention**: 365 days (regulatory compliance)  
**Use Cases**: Failure investigation, performance tracking, SLA monitoring

**Key Functions**:
* `log_pipeline_start()` - Initialize run tracking
* `log_pipeline_success()` - Record successful completion
* `log_pipeline_failure()` - Capture errors and stack traces
* `get_recent_runs()` - Query execution history

---

#### 2. `audit_dq_results`
**Purpose**: Track data quality rule evaluations and violations  
**Target Table**: `workspace.audit.audit_dq_results`  
**Captures**: Rule name, severity (ERROR/WARN/INFO), violation count, affected records, table/column, threshold, business impact  
**Triggers**: Called by Silver layer DQ validation notebooks  
**Retention**: 180 days  
**Use Cases**: Quality degradation alerts, compliance reporting, root cause analysis

**Key Functions**:
* `log_dq_check()` - Record rule evaluation result
* `aggregate_dq_summary()` - Daily quality scorecard
* `identify_quality_trends()` - Detect degradation patterns

---

#### 3. `audit_access_events`
**Purpose**: Log data access requests for security and compliance  
**Target Table**: `workspace.audit.audit_access_events`  
**Captures**: User ID, accessed table, query text (sanitized), timestamp, access granted/denied, IP address, session ID  
**Triggers**: Unity Catalog audit logs integration (if enabled)  
**Retention**: 730 days (security compliance)  
**Use Cases**: Security investigations, regulatory audits, access pattern analysis

**Key Functions**:
* `parse_uc_audit_logs()` - Extract relevant access events
* `detect_anomalous_access()` - Flag unusual access patterns
* `generate_compliance_report()` - Export audit trails

---

### **Operational Reporting**

#### 4. `audit_daily_summary`
**Purpose**: Generate daily operational summary reports  
**Target Table**: `workspace.audit.audit_daily_summary`  
**Captures**: Pipeline success rate, DQ error count, data freshness, processing latency, top failures  
**Schedule**: Runs daily at 6 AM UTC  
**Retention**: 90 days  
**Use Cases**: Daily standup reports, SLA tracking, capacity planning

**Key Metrics**:
* Pipeline success rate (target: >95%)
* DQ error-level violations (target: 0)
* Data freshness (target: <24 hours)
* Processing latency P95 (target: <30 min)

**Output**: 
* Persisted summary in audit table
* Email/Slack notification to data-ops-team
* Triggers critical alerts if SLA breached

---

#### 5. `audit_notification_dispatch`
**Purpose**: Dispatch operational alerts and reports via multiple channels  
**Notification Types**:
* **Critical Alerts** (Real-time): Pipeline failures, DQ ERROR-level breaches, security incidents
* **Daily Reports** (Scheduled): Daily summary, SLA compliance status, trends  
**Channels**: Email (SMTP), Slack webhooks, custom webhooks, PagerDuty integration  
**Configuration**: Uses Databricks Secrets (`lmip-notifications` scope)  
**Use Cases**: Incident response, operator awareness, executive reporting

**Key Functions**:
* `send_critical_alert()` - Real-time failure notifications
* `send_daily_report()` - Scheduled operational summaries
* `dispatch_to_channel()` - Multi-channel delivery
* `log_notification_audit()` - Track all dispatched notifications

**Notification Parameters**:
```python
# Example: Trigger critical alert
dbutils.notebook.run(
    "./audit_notification_dispatch",
    timeout_seconds=300,
    arguments={
        "notification_type": "critical_alert",
        "alert_severity": "HIGH",
        "alert_title": "Silver Layer Pipeline Failed",
        "alert_message": "silver_standardize_jobs failed with 523 DQ errors",
        "alert_context": json.dumps({"pipeline": "silver_standardize_jobs", "run_id": "abc123"}),
        "channel": "slack"
    }
)
```

---

## 🔧 Usage Patterns

### **Pattern 1: Pipeline Run Tracking**

Every pipeline notebook should call audit logging:

```python
# At notebook start
run_id, start_time = start_pipeline_run(
    notebook_name="bronze_dedupe_raw_payload",
    layer="BRONZE",
    dataset_name="job_snapshots",
    source_file="arbeitnow_batch_123.json"
)

try:
    # ... pipeline logic ...
    records_processed = df.count()
    
    # On success
    log_pipeline_success(run_id, start_time, records_processed)
except Exception as e:
    # On failure
    log_pipeline_failure(run_id, start_time, str(e), records_processed=0)
    raise
```

---

### **Pattern 2: Data Quality Logging**

Log DQ rule results after validation:

```python
from pyspark.sql.functions import col

# Run DQ check
null_violations = df.filter(col("company_name").isNull()).count()

# Log result
log_dq_check(
    rule_name="company_name_not_null",
    severity="ERROR",
    table_name="silver.silver_jobs_current",
    column_name="company_name",
    violation_count=null_violations,
    threshold=0,
    passed=(null_violations == 0)
)
```

---

### **Pattern 3: Query Audit History**

```sql
-- Recent pipeline failures
SELECT 
    pipeline_name,
    start_time,
    error_message,
    records_processed
FROM workspace.audit.audit_pipeline_runs
WHERE status = 'FAILED'
  AND start_time >= CURRENT_DATE() - INTERVAL 7 DAYS
ORDER BY start_time DESC;

-- DQ violation trends
SELECT 
    DATE(check_timestamp) AS check_date,
    rule_name,
    severity,
    SUM(violation_count) AS total_violations
FROM workspace.audit.audit_dq_results
WHERE check_timestamp >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY check_date, rule_name, severity
ORDER BY check_date DESC, total_violations DESC;
```

---

## 🔄 Data Flow & Dependencies

### **Audit Event Sources**

```
Ingestion Layer    →  audit_pipeline_runs (ingestion logs)
  ↓
Bronze Layer       →  audit_pipeline_runs (bronze logs)
  ↓
Silver Layer       →  audit_pipeline_runs + audit_dq_results
  ↓
Semantic Layer     →  audit_pipeline_runs (semantic logs)
  ↓
Warehouse Layer    →  audit_pipeline_runs (warehouse logs)
  ↓
Gold Layer         →  audit_pipeline_runs (gold logs)

Unity Catalog      →  audit_access_events (access logs)
```

### **Primary Audit Tables**

| Table | Purpose | Retention |
|-------|---------|----------|
| `audit.audit_pipeline_runs` | Pipeline execution history | 365 days |
| `audit.audit_dq_results` | Data quality violations | 180 days |
| `audit.audit_access_events` | Data access requests | 730 days |
| `audit.audit_daily_summary` | Daily operational reports | 90 days |
| `audit.audit_notification_log` | Dispatched notifications | 90 days |

### **Notification Dependencies**

* **Databricks Secrets**: `lmip-notifications` scope
  * `smtp_server`, `smtp_port`, `email_from`, `email_password`
  * `slack_webhook`, `custom_webhook`
* **External Services**: SMTP server, Slack workspace, webhook endpoints

---

## 🚨 Monitoring & Alerts

### **SLA Definitions**

| Metric | Target | Alert Threshold |
|--------|--------|----------------|
| Pipeline Success Rate | >95% | <90% |
| DQ ERROR-level Violations | 0 | >0 |
| Data Freshness | <24 hours | >36 hours |
| Processing Latency (P95) | <30 min | >60 min |
| Critical Alert Response Time | <15 min | >30 min |

### **Alert Severity Levels**

* **HIGH** - Immediate action required (pipeline failure, DQ ERROR, security incident)
* **MEDIUM** - Review within 4 hours (DQ WARN, SLA near breach, resource constraints)
* **LOW** - Review within 24 hours (informational, trends, recommendations)

### **Notification Channels**

* **Email** - data-ops-team@company.com (all severities)
* **Slack** - #data-pipeline-alerts (HIGH/MEDIUM only)
* **PagerDuty** - On-call rotation (HIGH only, 24/7)

### **Example Monitoring Queries**

```sql
-- Pipeline success rate (last 24 hours)
SELECT 
    ROUND(100.0 * SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) / COUNT(*), 2) AS success_rate
FROM workspace.audit.audit_pipeline_runs
WHERE start_time >= CURRENT_TIMESTAMP() - INTERVAL 24 HOURS;

-- Active DQ ERROR-level violations
SELECT 
    rule_name,
    table_name,
    SUM(violation_count) AS total_violations,
    MAX(check_timestamp) AS last_seen
FROM workspace.audit.audit_dq_results
WHERE severity = 'ERROR'
  AND check_timestamp >= CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY rule_name, table_name
HAVING total_violations > 0;
```

---

## ✅ Best Practices

### **Audit Logging Guidelines**

1. **Always Log Pipeline Start** - Even if notebook fails immediately
2. **Include Correlation IDs** - `run_id`, `batch_id` for end-to-end tracing
3. **Sanitize Error Messages** - Remove PII, credentials, sensitive data
4. **Log Business Context** - Dataset name, source file, record counts
5. **Use Structured Logging** - Consistent field names across all notebooks
6. **Fail Gracefully** - Audit logging failures should not block pipelines

### **Data Quality Logging**

* **Log ALL Rule Evaluations** - Not just failures (trend analysis needs full history)
* **Include Sample Violations** - First 10 violating record IDs for debugging
* **Set Reasonable Thresholds** - 0 for ERROR-level, tolerance for WARN-level
* **Document Business Impact** - Why this rule matters for downstream consumers

### **Security & Compliance**

* **Never Log Raw Query Text** - Sanitize user queries, remove literals
* **Respect Retention Policies** - Automated cleanup prevents unbounded growth
* **Encrypt Sensitive Fields** - Access patterns, IP addresses may be sensitive
* **Audit the Auditors** - Track who queries audit tables (meta-audit)

### **Notification Best Practices**

* **Avoid Alert Fatigue** - High severity only for actionable issues
* **Include Context** - Link to run ID, table name, error message in alerts
* **Test Notification Channels** - Verify SMTP, Slack, webhooks before production
* **Rate Limit Alerts** - Don't spam same alert repeatedly

---

## 🔧 Troubleshooting

### **Issue: Audit logs not appearing**

**Symptoms**: Pipeline runs not logged in `audit_pipeline_runs`  
**Causes**:
* Audit logging call failed silently
* Pipeline_Metadata_Manager not imported
* Audit table doesn't exist

**Solution**:
```python
# Verify audit table exists
spark.sql("DESCRIBE TABLE workspace.audit.audit_pipeline_runs")

# Check recent inserts
spark.sql("SELECT COUNT(*), MAX(start_time) FROM workspace.audit.audit_pipeline_runs")
```

---

### **Issue: Notifications not sending**

**Symptoms**: Alerts not received in email/Slack  
**Causes**:
* Databricks Secrets not configured
* SMTP credentials invalid
* Slack webhook URL incorrect
* Network egress blocked

**Solution**:
```python
# Test secret retrieval
try:
    smtp_server = dbutils.secrets.get(scope="lmip-notifications", key="smtp_server")
    print(f"SMTP server: {smtp_server}")
except Exception as e:
    print(f"Secret not found: {e}")

# Run notification test
dbutils.notebook.run(
    "./audit_notification_dispatch",
    300,
    {"notification_type": "daily_summary", "channel": "email"}
)
```

---

### **Issue: Audit tables growing too large**

**Symptoms**: Query performance degrading, storage costs increasing  
**Causes**: Retention cleanup not running

**Solution**:
```sql
-- Check table sizes
DESCRIBE DETAIL workspace.audit.audit_pipeline_runs;

-- Manually delete old records (outside retention)
DELETE FROM workspace.audit.audit_pipeline_runs
WHERE start_time < CURRENT_DATE() - INTERVAL 365 DAYS;

-- Optimize and vacuum
OPTIMIZE workspace.audit.audit_pipeline_runs;
VACUUM workspace.audit.audit_pipeline_runs RETAIN 168 HOURS;
```

---

## 📚 Related Documentation

* **Pipeline Metadata Manager** - See `notebooks/Pipeline_Metadata_Manager` for centralized audit logging functions
* **Data Quality Framework** - See `notebooks/silver/silver_dq_validate` for DQ rule definitions
* **Unity Catalog Audit** - See Databricks documentation for native audit log integration
* **Notification Setup** - See `deployment/secrets_setup.md` for Databricks Secrets configuration

---

## 🏁 Quick Start

### **1. Query Recent Pipeline Failures**
```sql
SELECT * FROM workspace.audit.audit_pipeline_runs
WHERE status = 'FAILED'
ORDER BY start_time DESC
LIMIT 10;
```

### **2. Send Test Notification**
```python
dbutils.notebook.run(
    "./audit_notification_dispatch",
    300,
    {"notification_type": "daily_summary", "channel": "slack"}
)
```

### **3. Generate Daily Summary**
```python
dbutils.notebook.run("./audit_daily_summary", 1800)
```

---

**Last Updated**: 2026-06-05  
**Maintained By**: Data Platform Team  
**On-Call**: Check PagerDuty schedule  
**Questions?**: Contact #data-ops on Slack